<a href="https://colab.research.google.com/github/Mario-Cattaneo/SemesterProject/blob/main/Eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Imports and Device Setup
import copy
import math
from dataclasses import dataclass, field
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing on device: {device}")

Executing on device: cuda


In [ ]:
# Cell 2: Hierarchical Configuration with Unified Time Units (ms & bits)
from dataclasses import dataclass, field
import math
from typing import Dict, List, Tuple
import numpy as np
import torch
import torch.nn as nn


@dataclass
class ChannelConfig:
    capacity_bits_per_ms: (
        float  # Capacity in bits/ms (e.g., 30 Mbps = 30,000 bits/ms)
    )
    prop_delay_ms: float  # Propagation delay in ms
    bit_width: int = 8  # b bits per scalar component

    @classmethod
    def from_mbps(cls, capacity_mbps: float, prop_delay_ms: float, bit_width=8):
        """Helper: converts Mbps to bits/ms (1 Mbps = 1,000 bits/ms)."""
        return cls(
            capacity_bits_per_ms=capacity_mbps * 1000.0,
            prop_delay_ms=prop_delay_ms,
            bit_width=bit_width,
        )


@dataclass
class NodeConfig:
    device_name: str
    exec_time_ms: float  # Device execution delay t_i^exec in ms


@dataclass
class ToleranceConfig:
    epsilon_ms: float = 0.8  # Global tolerance band [t_target - eps, t_target]
    epsilon_margin_ms: float = (
        0.3  # Max allowed SLA ceiling margin (t_target + margin)
    )


@dataclass
class ALMConfig:
    t_max: float  # Target SLA in ms
    lambda1_init: float = 0.0
    lambda2_init: float = 0.005
    rho: float = 1.5
    tau_viol: float = 0.25
    e_step: int = 2


@dataclass
class GateConfig:
    alpha: float = 10.0  # Smooth-max sharpness (constant)
    beta_init: float = 1.5
    beta_min: float = 0.05
    decay_rate: float = 0.90
    gamma: float = -0.1
    zeta: float = 1.1
    s_init: float = 1.0


@dataclass
class ExperimentConfig:
    experiment_name: str = "TBN_Component_Level"
    batch_size: int = 128
    epochs: int = 8
    lr_s: float = 0.05  # Learning rate for component gates
    lr_c: float = 0.01  # Learning rate for continuous imputations
    nodes: Dict[int, NodeConfig] = field(default_factory=dict)
    edges: Dict[Tuple[int, int], ChannelConfig] = field(default_factory=dict)
    alm: ALMConfig = field(default_factory=lambda: ALMConfig(t_max=25.0))
    gate: GateConfig = field(default_factory=GateConfig)
    tolerance: ToleranceConfig = field(default_factory=ToleranceConfig)


print("Cell 2: Unified configuration layer ready.")

Cell 2: Unified configuration layer ready.


In [ ]:
# Cell 3: True Component-Level (Element-Wise) Gating and Imputation


class EdgeBottleneck(nn.Module):
    """Component-level gating: learns one gate s and one imputation c for every

    individual scalar feature k in D_{i,j}.
    """

    def __init__(
        self,
        feature_shape: Tuple[int, ...],
        channel_cfg: ChannelConfig,
        gate_cfg: GateConfig,
    ):
        super().__init__()
        self.feature_shape = feature_shape  # e.g., (32, 32, 32)
        self.channel_cfg = channel_cfg
        self.gate_cfg = gate_cfg

        self.total_components = int(np.prod(feature_shape))

        # s and c match the EXACT tensor dimensions (C, H, W)
        self.s = nn.Parameter(torch.full(feature_shape, gate_cfg.s_init))
        self.c = nn.Parameter(torch.zeros(feature_shape))

    def get_transmission_prob(self, beta: float) -> torch.Tensor:
        """p(m_i,j^(k) > 0) evaluated component-wise."""
        offset = beta * math.log(-self.gate_cfg.gamma / self.gate_cfg.zeta)
        return torch.sigmoid(self.s - offset)

    def expected_transmission_time_ms(self, beta: float) -> torch.Tensor:
        """t_trans = (b * E[|Omega|]) / C [bits / (bits/ms) = ms]."""
        probs = self.get_transmission_prob(beta)
        expected_active_components = torch.sum(probs)
        total_bits = expected_active_components * self.channel_cfg.bit_width
        # Direct unified division: bits / (bits/ms) = ms
        trans_time_ms = total_bits / self.channel_cfg.capacity_bits_per_ms
        return trans_time_ms

    def forward(
        self, z: torch.Tensor, beta: float, deterministic: bool = False
    ) -> Tuple[torch.Tensor, float]:
        s_broad = self.s.unsqueeze(0)
        c_broad = self.c.unsqueeze(0)

        if not deterministic:
            eps = torch.rand_like(s_broad).clamp(1e-6, 1.0 - 1e-6)
            logits_eps = torch.log(eps) - torch.log(1.0 - eps)
            m_tilde = torch.sigmoid((logits_eps + s_broad) / beta)
            m_bar = (
                m_tilde * (self.gate_cfg.zeta - self.gate_cfg.gamma)
                + self.gate_cfg.gamma
            )
            m = torch.clamp(m_bar, 0.0, 1.0)

            # u^(k) = z^(k) * m^(k) + (1 - m^(k)) * c^(k)
            u = z * m + (1.0 - m) * c_broad
            active_fraction = float(m.mean().item())
            return u, active_fraction
        else:
            threshold = beta * math.log(
                -self.gate_cfg.gamma / self.gate_cfg.zeta
            )
            hard_mask = (s_broad >= threshold).float()

            u = z * hard_mask + (1.0 - hard_mask) * c_broad
            active_fraction = float(hard_mask.mean().item())
            return u, active_fraction


print("Cell 3: Component-level EdgeBottleneck compiled.")

Cell 3: Component-level EdgeBottleneck compiled.


In [ ]:
# Cell 4: DAGTimingEngine with Direct Millisecond Summation


class DAGTimingEngine:

    def __init__(
        self,
        nodes: Dict[int, NodeConfig],
        edges: Dict[Tuple[int, int], ChannelConfig],
    ):
        self.nodes = nodes
        self.edges = edges

        self.parents: Dict[int, List[int]] = {i: [] for i in nodes}
        self.children: Dict[int, List[int]] = {i: [] for i in nodes}
        for u, v in edges.keys():
            self.parents[v].append(u)
            self.children[u].append(v)

        self.topo_order = self._get_topological_sort()
        self.terminal_node = self.topo_order[-1]

    def _get_topological_sort(self) -> List[int]:
        in_deg = {i: len(self.parents[i]) for i in self.nodes}
        queue = [i for i in self.nodes if in_deg[i] == 0]
        order = []
        while queue:
            curr = queue.pop(0)
            order.append(curr)
            for nxt in self.children[curr]:
                in_deg[nxt] -= 1
                if in_deg[nxt] == 0:
                    queue.append(nxt)
        return order

    def compute_relaxed_latency(
        self,
        bottlenecks: Dict[Tuple[int, int], EdgeBottleneck],
        beta: float,
        alpha: float,
    ) -> torch.Tensor:
        t_tilde: Dict[int, torch.Tensor] = {}

        for j in self.topo_order:
            t_exec = torch.tensor(
                self.nodes[j].exec_time_ms, device=bottlenecks[(0, 1)].s.device
            )

            if len(self.parents[j]) == 0:
                t_tilde[j] = t_exec
            else:
                branch_arrivals = []
                for i in self.parents[j]:
                    t_trans = bottlenecks[(i, j)].expected_transmission_time_ms(
                        beta
                    )
                    t_prop = torch.tensor(
                        self.edges[(i, j)].prop_delay_ms, device=t_trans.device
                    )
                    z_ij = (
                        t_tilde[i] + t_prop + t_trans
                    )  # All terms strictly in ms
                    branch_arrivals.append(z_ij)

                stacked = torch.stack(branch_arrivals)
                z_max = torch.max(stacked).detach()
                lse = z_max + (1.0 / alpha) * torch.log(
                    torch.sum(torch.exp(alpha * (stacked - z_max)))
                )
                t_tilde[j] = t_exec + lse

        return t_tilde[self.terminal_node]

    def compute_physical_latency(
        self,
        bottlenecks: Dict[Tuple[int, int], EdgeBottleneck],
        beta: float,
        deterministic: bool = True,
    ) -> float:
        t_physical: Dict[int, float] = {}

        for j in self.topo_order:
            t_exec = self.nodes[j].exec_time_ms

            if len(self.parents[j]) == 0:
                t_physical[j] = t_exec
            else:
                branch_delays = []
                for i in self.parents[j]:
                    bneck = bottlenecks[(i, j)]
                    if deterministic:
                        thresh = beta * math.log(
                            -bneck.gate_cfg.gamma / bneck.gate_cfg.zeta
                        )
                        active_components = (bneck.s >= thresh).sum().item()
                    else:
                        active_components = (
                            bneck.get_transmission_prob(beta).sum().item()
                        )

                    total_bits = (
                        active_components * bneck.channel_cfg.bit_width
                    )
                    # bits / (bits/ms) = ms
                    t_trans = (
                        total_bits / self.edges[(i, j)].capacity_bits_per_ms
                    )
                    t_prop = self.edges[(i, j)].prop_delay_ms
                    branch_delays.append(t_physical[i] + t_prop + t_trans)

                t_physical[j] = t_exec + max(branch_delays)

        return t_physical[self.terminal_node]

    def compute_irreducible_latency(self) -> float:
        """Physical floor when t_trans -> 0 (ms)."""
        t_floor: Dict[int, float] = {}
        for j in self.topo_order:
            t_exec = self.nodes[j].exec_time_ms
            if len(self.parents[j]) == 0:
                t_floor[j] = t_exec
            else:
                branch_delays = [
                    t_floor[i] + self.edges[(i, j)].prop_delay_ms
                    for i in self.parents[j]
                ]
                t_floor[j] = t_exec + max(branch_delays)
        return t_floor[self.terminal_node]


print("Cell 4: DAGTimingEngine ready.")

Cell 4: DAGTimingEngine ready.


In [ ]:
# ==============================================================================
# Cell 5: Dynamic Modular Architecture for 4 Extreme Topologies
# ==============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch=None, stride=1):
        super().__init__()
        out_ch = out_ch or in_ch
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.skip = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )

    def forward(self, x):
        return F.relu(self.skip(x) + self.conv(x))


class DynamicSplitDAG(nn.Module):
    """Modular Collaborative ResNet supporting all 4 extreme topological graphs."""
    def __init__(self, topo_type: str, gate_cfg: GateConfig, channel_cfgs: dict):
        super().__init__()
        self.topo_type = topo_type.lower()
        self.gate_cfg = gate_cfg
        self.channel_cfgs = channel_cfgs
        self.node_splits = nn.ModuleDict()
        self.bottlenecks = nn.ModuleDict()

        # ----------------------------------------------------------------------
        # 1. DIAMOND (4 Nodes)
        # ----------------------------------------------------------------------
        if self.topo_type == "diamond":
            self.node_splits["0"] = nn.Sequential(
                nn.Conv2d(3, 64, 3, 1, 1, bias=False), nn.BatchNorm2d(64), nn.ReLU(), ResBlock(64)
            )
            self.node_splits["1"] = ResBlock(32, 64, stride=2)
            self.node_splits["2"] = ResBlock(32, 64, stride=2)
            self.node_splits["3"] = nn.Sequential(
                ResBlock(128), nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(), nn.Linear(128, 10)
            )
            self.bottlenecks["0->1"] = EdgeBottleneck((32, 32, 32), channel_cfgs[(0, 1)], gate_cfg)
            self.bottlenecks["0->2"] = EdgeBottleneck((32, 32, 32), channel_cfgs[(0, 2)], gate_cfg)
            self.bottlenecks["1->3"] = EdgeBottleneck((64, 16, 16), channel_cfgs[(1, 3)], gate_cfg)
            self.bottlenecks["2->3"] = EdgeBottleneck((64, 16, 16), channel_cfgs[(2, 3)], gate_cfg)

        # ----------------------------------------------------------------------
        # 2. DEEP ASYMMETRIC (10 Nodes: Path A = 3 hops, Path B = 5 hops)
        # ----------------------------------------------------------------------
        elif self.topo_type == "deep_asymmetric":
            # Node 0 (Stem 3->64, split 32+32)
            self.node_splits["0"] = nn.Sequential(
                nn.Conv2d(3, 64, 3, 1, 1, bias=False), nn.BatchNorm2d(64), nn.ReLU(), ResBlock(64)
            )
            # Path A (Nodes 1 -> 2 -> 3)
            self.node_splits["1"] = ResBlock(32)
            self.node_splits["2"] = ResBlock(32, 64, stride=2)
            self.node_splits["3"] = ResBlock(64)

            # Path B (Nodes 4 -> 5 -> 6 -> 7 -> 8)
            self.node_splits["4"] = ResBlock(32)
            self.node_splits["5"] = ResBlock(32)
            self.node_splits["6"] = ResBlock(32, 64, stride=2)
            self.node_splits["7"] = ResBlock(64)
            self.node_splits["8"] = ResBlock(64)

            # Node 9 (Terminal Sink: 64 + 64 = 128 channels)
            self.node_splits["9"] = nn.Sequential(
                ResBlock(128), nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(), nn.Linear(128, 10)
            )
            # Bottlenecks for Path A
            self.bottlenecks["0->1"] = EdgeBottleneck((32, 32, 32), channel_cfgs[(0, 1)], gate_cfg)
            self.bottlenecks["1->2"] = EdgeBottleneck((32, 32, 32), channel_cfgs[(1, 2)], gate_cfg)
            self.bottlenecks["2->3"] = EdgeBottleneck((64, 16, 16), channel_cfgs[(2, 3)], gate_cfg)
            self.bottlenecks["3->9"] = EdgeBottleneck((64, 16, 16), channel_cfgs[(3, 9)], gate_cfg)
            # Bottlenecks for Path B
            self.bottlenecks["0->4"] = EdgeBottleneck((32, 32, 32), channel_cfgs[(0, 4)], gate_cfg)
            self.bottlenecks["4->5"] = EdgeBottleneck((32, 32, 32), channel_cfgs[(4, 5)], gate_cfg)
            self.bottlenecks["5->6"] = EdgeBottleneck((32, 32, 32), channel_cfgs[(5, 6)], gate_cfg)
            self.bottlenecks["6->7"] = EdgeBottleneck((64, 16, 16), channel_cfgs[(6, 7)], gate_cfg)
            self.bottlenecks["7->8"] = EdgeBottleneck((64, 16, 16), channel_cfgs[(7, 8)], gate_cfg)
            self.bottlenecks["8->9"] = EdgeBottleneck((64, 16, 16), channel_cfgs[(8, 9)], gate_cfg)

        # ----------------------------------------------------------------------
        # 3. WIDE SHALLOW (7 Nodes: 1 Ingress -> 5 Workers -> 1 Sink)
        # ----------------------------------------------------------------------
        elif self.topo_type == "wide_shallow":
            # Node 0 (Stem 3->60, split into 5 x 12 channels)
            self.node_splits["0"] = nn.Sequential(
                nn.Conv2d(3, 60, 3, 1, 1, bias=False), nn.BatchNorm2d(60), nn.ReLU(), ResBlock(60)
            )
            # 5 Parallel Workers: 12 in -> 24 out (stride 2)
            for i in range(1, 6):
                self.node_splits[str(i)] = ResBlock(12, 24, stride=2)
                self.bottlenecks[f"0->{i}"] = EdgeBottleneck((12, 32, 32), channel_cfgs[(0, i)], gate_cfg)
                self.bottlenecks[f"{i}->6"] = EdgeBottleneck((24, 16, 16), channel_cfgs[(i, 6)], gate_cfg)

            # Node 6 (Sink: 5 x 24 = 120 channels in)
            self.node_splits["6"] = nn.Sequential(
                ResBlock(120), nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(), nn.Linear(120, 10)
            )

        # ----------------------------------------------------------------------
        # 4. WIDE DEEP (9 Nodes: 3 Multi-Hop Paths -> Sink)
        # ----------------------------------------------------------------------
        elif self.topo_type == "wide_deep":
            # Node 0 (Stem 3->60, split into 3 x 20 channels)
            self.node_splits["0"] = nn.Sequential(
                nn.Conv2d(3, 60, 3, 1, 1, bias=False), nn.BatchNorm2d(60), nn.ReLU(), ResBlock(60)
            )
            # Path A (2 hops: 1 -> 2)
            self.node_splits["1"] = ResBlock(20, 40)
            self.node_splits["2"] = ResBlock(40, 40, stride=2)
            # Path B (3 hops: 3 -> 4 -> 5)
            self.node_splits["3"] = ResBlock(20, 20)
            self.node_splits["4"] = ResBlock(20, 40)
            self.node_splits["5"] = ResBlock(40, 40, stride=2)
            # Path C (2 hops: 6 -> 7)
            self.node_splits["6"] = ResBlock(20, 40)
            self.node_splits["7"] = ResBlock(40, 40, stride=2)

            # Node 8 (Sink: 40 + 40 + 40 = 120 channels in)
            self.node_splits["8"] = nn.Sequential(
                ResBlock(120), nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(), nn.Linear(120, 10)
            )
            # Bottlenecks
            self.bottlenecks["0->1"] = EdgeBottleneck((20, 32, 32), channel_cfgs[(0, 1)], gate_cfg)
            self.bottlenecks["1->2"] = EdgeBottleneck((40, 32, 32), channel_cfgs[(1, 2)], gate_cfg)
            self.bottlenecks["2->8"] = EdgeBottleneck((40, 16, 16), channel_cfgs[(2, 8)], gate_cfg)

            self.bottlenecks["0->3"] = EdgeBottleneck((20, 32, 32), channel_cfgs[(0, 3)], gate_cfg)
            self.bottlenecks["3->4"] = EdgeBottleneck((20, 32, 32), channel_cfgs[(3, 4)], gate_cfg)
            self.bottlenecks["4->5"] = EdgeBottleneck((40, 32, 32), channel_cfgs[(4, 5)], gate_cfg)
            self.bottlenecks["5->8"] = EdgeBottleneck((40, 16, 16), channel_cfgs[(5, 8)], gate_cfg)

            self.bottlenecks["0->6"] = EdgeBottleneck((20, 32, 32), channel_cfgs[(0, 6)], gate_cfg)
            self.bottlenecks["6->7"] = EdgeBottleneck((40, 32, 32), channel_cfgs[(6, 7)], gate_cfg)
            self.bottlenecks["7->8"] = EdgeBottleneck((40, 16, 16), channel_cfgs[(7, 8)], gate_cfg)

    def get_bottleneck_map(self):
        return {
            edge: self.bottlenecks[f"{edge[0]}->{edge[1]}"]
            for edge in self.channel_cfgs.keys()
            if f"{edge[0]}->{edge[1]}" in self.bottlenecks
        }

    def forward(self, x: torch.Tensor, beta: float, deterministic: bool = False):
        if self.topo_type == "diamond":
            z0 = self.node_splits["0"](x)
            z0_A, z0_B = torch.split(z0, 32, dim=1)
            u01, _ = self.bottlenecks["0->1"](z0_A, beta, deterministic)
            u02, _ = self.bottlenecks["0->2"](z0_B, beta, deterministic)
            z1 = self.node_splits["1"](u01)
            z2 = self.node_splits["2"](u02)
            u13, _ = self.bottlenecks["1->3"](z1, beta, deterministic)
            u23, _ = self.bottlenecks["2->3"](z2, beta, deterministic)
            return self.node_splits["3"](torch.cat([u13, u23], dim=1))

        elif self.topo_type == "deep_asymmetric":
            z0 = self.node_splits["0"](x)
            z0_A, z0_B = torch.split(z0, 32, dim=1)
            # Path A
            u01, _ = self.bottlenecks["0->1"](z0_A, beta, deterministic)
            z1 = self.node_splits["1"](u01)
            u12, _ = self.bottlenecks["1->2"](z1, beta, deterministic)
            z2 = self.node_splits["2"](u12)
            u23, _ = self.bottlenecks["2->3"](z2, beta, deterministic)
            z3 = self.node_splits["3"](u23)
            u39, _ = self.bottlenecks["3->9"](z3, beta, deterministic)
            # Path B
            u04, _ = self.bottlenecks["0->4"](z0_B, beta, deterministic)
            z4 = self.node_splits["4"](u04)
            u45, _ = self.bottlenecks["4->5"](z4, beta, deterministic)
            z5 = self.node_splits["5"](u45)
            u56, _ = self.bottlenecks["5->6"](z5, beta, deterministic)
            z6 = self.node_splits["6"](u56)
            u67, _ = self.bottlenecks["6->7"](z6, beta, deterministic)
            z7 = self.node_splits["7"](u67)
            u78, _ = self.bottlenecks["7->8"](z7, beta, deterministic)
            z8 = self.node_splits["8"](u78)
            u89, _ = self.bottlenecks["8->9"](z8, beta, deterministic)
            return self.node_splits["9"](torch.cat([u39, u89], dim=1))

        elif self.topo_type == "wide_shallow":
            z0 = self.node_splits["0"](x)
            slices = torch.split(z0, 12, dim=1)
            worker_outs = []
            for i in range(1, 6):
                u_in, _ = self.bottlenecks[f"0->{i}"](slices[i-1], beta, deterministic)
                z_w = self.node_splits[str(i)](u_in)
                u_out, _ = self.bottlenecks[f"{i}->6"](z_w, beta, deterministic)
                worker_outs.append(u_out)
            return self.node_splits["6"](torch.cat(worker_outs, dim=1))

        elif self.topo_type == "wide_deep":
            z0 = self.node_splits["0"](x)
            zA, zB, zC = torch.split(z0, 20, dim=1)
            # Path A
            u01, _ = self.bottlenecks["0->1"](zA, beta, deterministic)
            z1 = self.node_splits["1"](u01)
            u12, _ = self.bottlenecks["1->2"](z1, beta, deterministic)
            z2 = self.node_splits["2"](u12)
            u28, _ = self.bottlenecks["2->8"](z2, beta, deterministic)
            # Path B
            u03, _ = self.bottlenecks["0->3"](zB, beta, deterministic)
            z3 = self.node_splits["3"](u03)
            u34, _ = self.bottlenecks["3->4"](z3, beta, deterministic)
            z4 = self.node_splits["4"](u34)
            u45, _ = self.bottlenecks["4->5"](z4, beta, deterministic)
            z5 = self.node_splits["5"](u45)
            u58, _ = self.bottlenecks["5->8"](z5, beta, deterministic)
            # Path C
            u06, _ = self.bottlenecks["0->6"](zC, beta, deterministic)
            z6 = self.node_splits["6"](u06)
            u67, _ = self.bottlenecks["6->7"](z6, beta, deterministic)
            z7 = self.node_splits["7"](u67)
            u78, _ = self.bottlenecks["7->8"](z7, beta, deterministic)
            return self.node_splits["8"](torch.cat([u28, u58, u78], dim=1))

print("Cell 5: DynamicSplitDAG compiled for all 4 extreme topologies.")

Cell 5: DynamicSplitDAG compiled for all 4 extreme topologies.


In [ ]:
# ==============================================================================
# Cell 6: Fast Dataset Setup + High-Accuracy Backbone Population (100 Epochs)
# ==============================================================================
import os
import shutil
import tarfile
import urllib.request
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download

assert torch.cuda.is_available(), "GPU disconnected! Ensure T4 GPU runtime is active."
device = torch.device("cuda")

# ------------------------------------------------------------------------------
# 0. Ensure Fast CIFAR-10 DataLoaders Exist
# ------------------------------------------------------------------------------
local_cifar_dir = "/content/cifar10_local"
target_batch_dir = os.path.join(local_cifar_dir, "cifar-10-batches-py")

if not os.path.exists(target_batch_dir):
    os.makedirs(local_cifar_dir, exist_ok=True)
    tar_archive_path = "/content/cifar-10-python.tar.gz"
    cdn_url = "https://huggingface.co/datasets/Shiym/ViT-FineTune/resolve/main/cifar-10-python.tar.gz"
    if not os.path.exists(tar_archive_path):
        print(">>> Downloading CIFAR-10 from CDN mirror (~2s at 80 MB/s)...")
        urllib.request.urlretrieve(cdn_url, tar_archive_path)
    print(">>> Extracting dataset...")
    with tarfile.open(tar_archive_path, "r:gz") as tar:
        tar.extractall(path=local_cifar_dir)
    print(">>> CIFAR-10 ready on fast local disk!")
else:
    print(">>> CIFAR-10 already cached locally.")

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

trainset = torchvision.datasets.CIFAR10(root=local_cifar_dir, train=True, download=False, transform=transform_train)
testset = torchvision.datasets.CIFAR10(root=local_cifar_dir, train=False, download=False, transform=transform_test)

train_loader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
test_loader = torch.utils.data.DataLoader(testset, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
print(f">>> DataLoaders ready: {len(trainset)} train, {len(testset)} test samples.\n")

# ------------------------------------------------------------------------------
# 1. Credentials & Repository Setup
# ------------------------------------------------------------------------------
HF_REPO_ID = "marioIsGoated/SemProjSplits"
LOCAL_CACHE_DIR = "/content/hf_backbones_cache"
os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)

try:
    HF_WRITE_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_WRITE_TOKEN = None

api = HfApi()
if HF_WRITE_TOKEN:
    try:
        api.create_repo(
            repo_id=HF_REPO_ID,
            repo_type="model",
            private=False,
            exist_ok=True,
            token=HF_WRITE_TOKEN
        )
        print(f">>> [HF Hub] Connected to repository: https://huggingface.co/{HF_REPO_ID}")
    except Exception as e:
        print(f">>> [Notice] Repo verification: {e}")
else:
    print(">>> [Warning] HF_WRITE_TOKEN not found. Models will be saved locally but NOT pushed to HF.")

# ------------------------------------------------------------------------------
# 2. Pretraining Configuration (100 Epochs for SOTA Backbone)
# ------------------------------------------------------------------------------
PRETRAIN_EPOCHS = 100
FORCE_RETRAIN = False   # <--- Set to True to overwrite the old 20-epoch checkpoints

# You can start with "diamond" (~6 mins) or retrain all 4
TOPOLOGIES_TO_PREPARE = [
    "diamond",
    # "deep_asymmetric",
    # "wide_shallow",
    # "wide_deep"
]

TOPO_EDGES = {
    "diamond": [(0, 1), (0, 2), (1, 3), (2, 3)],
    "deep_asymmetric": [
        (0, 1), (1, 2), (2, 3), (3, 9),
        (0, 4), (4, 5), (5, 6), (6, 7), (7, 8), (8, 9)
    ],
    "wide_shallow": [
        (0, 1), (0, 2), (0, 3), (0, 4), (0, 5),
        (1, 6), (2, 6), (3, 6), (4, 6), (5, 6)
    ],
    "wide_deep": [
        (0, 1), (1, 2), (2, 8),
        (0, 3), (3, 4), (4, 5), (5, 8),
        (0, 6), (6, 7), (7, 8)
    ]
}

# ------------------------------------------------------------------------------
# 3. Active Check, Train, and Upload Loop
# ------------------------------------------------------------------------------
print("\n" + "=" * 80)
print(f">>> POPULATING BACKBONES ({len(TOPOLOGIES_TO_PREPARE)} TOPOLOGIES | {PRETRAIN_EPOCHS} EPOCHS)")
print(f">>> Force Retrain: {FORCE_RETRAIN}")
print("=" * 80)

for topo_key in TOPOLOGIES_TO_PREPARE:
    fname = f"backbone_{topo_key}.pt"
    local_path = os.path.join(LOCAL_CACHE_DIR, fname)
    is_on_hf = False

    print(f"\n--> Checking '{topo_key}' ({fname})...")

    # 1. Download if exists and not forcing retrain
    if not FORCE_RETRAIN:
        try:
            downloaded = hf_hub_download(
                repo_id=HF_REPO_ID,
                filename=fname,
                cache_dir=LOCAL_CACHE_DIR,
                token=HF_WRITE_TOKEN
            )
            shutil.copy(downloaded, local_path)
            is_on_hf = True
            print(f"    [Found on HF] Downloaded '{fname}' successfully!")
        except Exception:
            print(f"    [Missing on HF] '{fname}' not found in remote repository.")

    # 2. Train from scratch using high-accuracy SGD recipe
    if not is_on_hf:
        print(f"    [Pretraining] Starting {PRETRAIN_EPOCHS} epochs for '{topo_key}' on T4 GPU (FP16 AMP)...")

        dummy_channel_cfgs = {
            edge: ChannelConfig.from_mbps(25.0, 1.0) for edge in TOPO_EDGES[topo_key]
        }
        gate_cfg = GateConfig(s_init=10.0)
        model = DynamicSplitDAG(topo_key, gate_cfg, dummy_channel_cfgs).to(device)

        # Gates wide open
        for bneck in model.get_bottleneck_map().values():
            bneck.s.requires_grad = False
            bneck.c.requires_grad = False
            bneck.s.data.fill_(10.0)

        # Standard CIFAR-10 ResNet Optimizer Recipe
        opt = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4, nesterov=True)
        sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=PRETRAIN_EPOCHS, eta_min=1e-4)
        crit = nn.CrossEntropyLoss()
        scaler = torch.amp.GradScaler('cuda')

        for ep in range(PRETRAIN_EPOCHS):
            model.train()
            train_correct, train_total = 0, 0

            for x, y in train_loader:
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                opt.zero_grad(set_to_none=True)

                with torch.amp.autocast('cuda'):
                    y_hat = model(x, beta=1.0, deterministic=True)
                    loss = crit(y_hat, y)

                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()

                train_correct += (y_hat.argmax(dim=1) == y).sum().item()
                train_total += x.size(0)

            sched.step()

            # Evaluate both Train & Unseen Test Accuracy
            if (ep + 1) % 10 == 0 or ep == PRETRAIN_EPOCHS - 1 or ep == 0:
                model.eval()
                test_correct, test_total = 0, 0
                with torch.no_grad():
                    for x_t, y_t in test_loader:
                        x_t, y_t = x_t.to(device, non_blocking=True), y_t.to(device, non_blocking=True)
                        with torch.amp.autocast('cuda'):
                            preds = model(x_t, beta=1.0, deterministic=True)
                        test_correct += (preds.argmax(dim=1) == y_t).sum().item()
                        test_total += x_t.size(0)

                train_acc = 100.0 * train_correct / train_total
                test_acc = 100.0 * test_correct / test_total
                print(f"        [Ep {ep+1:03d}/{PRETRAIN_EPOCHS}] Train Acc: {train_acc:5.2f}% | Test Acc: {test_acc:5.2f}% | LR: {sched.get_last_lr()[0]:.5f}")

        torch.save(model.state_dict(), local_path)
        print(f"    [Local Save] Cached at '{local_path}'")

        # 3. Overwrite file on Hugging Face
        if HF_WRITE_TOKEN:
            print(f"    [Uploading] Pushing '{fname}' to Hugging Face...")
            try:
                api.upload_file(
                    path_or_fileobj=local_path,
                    path_in_repo=fname,
                    repo_id=HF_REPO_ID,
                    token=HF_WRITE_TOKEN
                )
                print(f"    >>> [SUCCESS] Uploaded '{fname}' to https://huggingface.co/{HF_REPO_ID}")
            except Exception as e:
                print(f"    >>> [Upload Error] Failed to push to HF: {e}")
        else:
            print("    [Notice] HF_WRITE_TOKEN not available. Skipped remote upload.")

print("\n" + "=" * 80)
print(">>> BACKBONE POPULATION COMPLETE!")
print("=" * 80)

>>> Downloading CIFAR-10 from CDN mirror (~2s at 80 MB/s)...
>>> Extracting dataset...


/tmp/ipykernel_2834/2455656875.py:34: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=local_cifar_dir)


>>> CIFAR-10 ready on fast local disk!
>>> DataLoaders ready: 50000 train, 10000 test samples.

>>> [Warning] HF_WRITE_TOKEN not found. Models will be saved locally but NOT pushed to HF.

>>> POPULATING BACKBONES (1 TOPOLOGIES | 100 EPOCHS)
>>> Force Retrain: False

--> Checking 'diamond' (backbone_diamond.pt)...


backbone_diamond.pt: reconstructing file:   0%|          |  0.00B / 2.77MB            

backbone_diamond.pt: downloading bytes:           |  0.00B            

    [Found on HF] Downloaded 'backbone_diamond.pt' successfully!

>>> BACKBONE POPULATION COMPLETE!


In [ ]:
# ==============================================================================
# Cell 6b: Hardware Profiles & Latency Floor/Ceiling Inspector
# ==============================================================================
CONFIG_REGISTRY = {
    # --------------------------------------------------------------------------
    # 1. DIAMOND (4 Nodes)
    # --------------------------------------------------------------------------
    "diamond_case4_dual_skew": {
        "topo_type": "diamond",
        "nodes": {
            0: NodeConfig("Ingress", exec_time_ms=3.0),
            1: NodeConfig("Worker_A_Fast", exec_time_ms=1.5),
            2: NodeConfig("Worker_B_Slow", exec_time_ms=9.5),
            3: NodeConfig("Terminal", exec_time_ms=2.0),
        },
        "edges": {
            (0, 1): ChannelConfig.from_mbps(30.0, 1.0),
            (0, 2): ChannelConfig.from_mbps(30.0, 1.0),
            (1, 3): ChannelConfig.from_mbps(60.0, 1.0),
            (2, 3): ChannelConfig.from_mbps(3.0, 1.0),  # Bottleneck
        },
    },

    # --------------------------------------------------------------------------
    # 2. DEEP ASYMMETRIC (10 Nodes: Path A = 3 hops, Path B = 5 hops)
    # --------------------------------------------------------------------------
    "deep_asymmetric_3vs5hops": {
        "topo_type": "deep_asymmetric",
        "nodes": {
            0: NodeConfig("Ingress", exec_time_ms=3.0),
            # Path A (Fast GPU nodes: 1.5ms each)
            1: NodeConfig("A1", exec_time_ms=1.5),
            2: NodeConfig("A2", exec_time_ms=1.5),
            3: NodeConfig("A3", exec_time_ms=1.5),
            # Path B (Slow CPU nodes: 4.0ms each)
            4: NodeConfig("B1", exec_time_ms=4.0),
            5: NodeConfig("B2", exec_time_ms=4.0),
            6: NodeConfig("B3", exec_time_ms=4.0),
            7: NodeConfig("B4", exec_time_ms=4.0),
            8: NodeConfig("B5", exec_time_ms=4.0),
            9: NodeConfig("Terminal", exec_time_ms=2.0),
        },
        "edges": {
            (0, 1): ChannelConfig.from_mbps(30.0, 1.0),
            (1, 2): ChannelConfig.from_mbps(30.0, 1.0),
            (2, 3): ChannelConfig.from_mbps(30.0, 1.0),
            (3, 9): ChannelConfig.from_mbps(50.0, 1.0),

            (0, 4): ChannelConfig.from_mbps(25.0, 1.0),
            (4, 5): ChannelConfig.from_mbps(20.0, 1.0),
            (5, 6): ChannelConfig.from_mbps(15.0, 1.0),
            (6, 7): ChannelConfig.from_mbps(10.0, 1.0),
            (7, 8): ChannelConfig.from_mbps(4.0, 1.0),   # Severe internal bottleneck
            (8, 9): ChannelConfig.from_mbps(20.0, 1.0),
        },
    },

    # --------------------------------------------------------------------------
    # 3. WIDE SHALLOW (7 Nodes: 5 Parallel Workers)
    # --------------------------------------------------------------------------
    "wide_shallow_5workers": {
        "topo_type": "wide_shallow",
        "nodes": {
            0: NodeConfig("Dispatcher", exec_time_ms=2.0),
            1: NodeConfig("W1_Fast", exec_time_ms=1.5),
            2: NodeConfig("W2_Mid", exec_time_ms=2.5),
            3: NodeConfig("W3_Mid", exec_time_ms=4.0),
            4: NodeConfig("W4_Slow", exec_time_ms=6.0),
            5: NodeConfig("W5_Bottleneck", exec_time_ms=10.0),
            6: NodeConfig("Aggregator", exec_time_ms=2.0),
        },
        "edges": {
            (0, 1): ChannelConfig.from_mbps(50.0, 1.0),
            (0, 2): ChannelConfig.from_mbps(40.0, 1.0),
            (0, 3): ChannelConfig.from_mbps(30.0, 1.0),
            (0, 4): ChannelConfig.from_mbps(20.0, 1.0),
            (0, 5): ChannelConfig.from_mbps(15.0, 1.0),

            (1, 6): ChannelConfig.from_mbps(60.0, 1.0),
            (2, 6): ChannelConfig.from_mbps(40.0, 1.0),
            (3, 6): ChannelConfig.from_mbps(25.0, 1.0),
            (4, 6): ChannelConfig.from_mbps(10.0, 1.0),
            (5, 6): ChannelConfig.from_mbps(3.0, 1.0),   # Extreme bottleneck link
        },
    },

    # --------------------------------------------------------------------------
    # 4. WIDE DEEP (9 Nodes: 3 Parallel Multi-Hop Paths)
    # --------------------------------------------------------------------------
    "wide_deep_3paths": {
        "topo_type": "wide_deep",
        "nodes": {
            0: NodeConfig("Ingress", exec_time_ms=2.0),
            # Path A (2 hops)
            1: NodeConfig("A1", exec_time_ms=2.0),
            2: NodeConfig("A2", exec_time_ms=2.0),
            # Path B (3 hops)
            3: NodeConfig("B1", exec_time_ms=3.0),
            4: NodeConfig("B2", exec_time_ms=4.0),
            5: NodeConfig("B3", exec_time_ms=5.0),
            # Path C (2 hops)
            6: NodeConfig("C1", exec_time_ms=2.5),
            7: NodeConfig("C2", exec_time_ms=2.5),
            8: NodeConfig("Sink", exec_time_ms=2.0),
        },
        "edges": {
            (0, 1): ChannelConfig.from_mbps(30.0, 1.0),
            (1, 2): ChannelConfig.from_mbps(30.0, 1.0),
            (2, 8): ChannelConfig.from_mbps(40.0, 1.0),

            (0, 3): ChannelConfig.from_mbps(25.0, 1.0),
            (3, 4): ChannelConfig.from_mbps(15.0, 1.0),
            (4, 5): ChannelConfig.from_mbps(4.0, 1.0),   # Bottleneck link
            (5, 8): ChannelConfig.from_mbps(25.0, 1.0),

            (0, 6): ChannelConfig.from_mbps(30.0, 1.0),
            (6, 7): ChannelConfig.from_mbps(30.0, 1.0),
            (7, 8): ChannelConfig.from_mbps(40.0, 1.0),
        },
    },
}

# --- Calculate and Display Floor & Ceiling Latencies ---
print("\n" + "=" * 94)
print(f"{'Config ID':<26} | {'Topology':<16} | {'Floor (ms)':<12} | {'Ceiling (ms)':<14} | {'Dynamic Range':<12}")
print("=" * 94)

for cfg_id, cfg_data in CONFIG_REGISTRY.items():
    engine = DAGTimingEngine(cfg_data["nodes"], cfg_data["edges"])
    floor_lat = engine.compute_irreducible_latency()

    dummy_model = DynamicSplitDAG(cfg_data["topo_type"], GateConfig(s_init=10.0), cfg_data["edges"]).to(device)
    for b in dummy_model.get_bottleneck_map().values():
        b.s.data.fill_(10.0)

    ceiling_lat = engine.compute_physical_latency(dummy_model.get_bottleneck_map(), beta=0.05, deterministic=True)
    cfg_data["floor_lat"] = floor_lat
    cfg_data["ceiling_lat"] = ceiling_lat

    print(f"{cfg_id:<26} | {cfg_data['topo_type']:<16} | {floor_lat:<12.2f} | {ceiling_lat:<14.2f} | {ceiling_lat - floor_lat:<12.2f} ms")

print("=" * 94 + "\n")


Config ID                  | Topology         | Floor (ms)   | Ceiling (ms)   | Dynamic Range
diamond_case4_dual_skew    | diamond          | 16.50        | 68.93          | 52.43        ms
deep_asymmetric_3vs5hops   | deep_asymmetric  | 31.00        | 124.50         | 93.50        ms
wide_shallow_5workers      | wide_shallow     | 16.00        | 38.94          | 22.94        ms
wide_deep_3paths           | wide_deep        | 20.00        | 122.67         | 102.67       ms



In [ ]:
# ==============================================================================
# Cell 8: Fully Normalized Pareto Sweep Suite (Dynamic Span [0, 1] Alignment)
# ==============================================================================
import copy
import gc
import json
import math
import os
import shutil
from dataclasses import dataclass, field
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

# ------------------------------------------------------------------------------
# 0. GPU Verification & Memory Cleanup
# ------------------------------------------------------------------------------
gc.collect()
torch.cuda.empty_cache()
assert torch.cuda.is_available(), "CRITICAL: GPU disconnected! Ensure GPU runtime is active."
device = torch.device("cuda")

# ------------------------------------------------------------------------------
# 1. MASTER USER CONFIGURATION (Fully Modifiable)
# ------------------------------------------------------------------------------
@dataclass
class ParetoSweepConfig:
    # 1. Dimensionless Tradeoff Multipliers (Omega is normalized to [0, 1])
    # ~0.05 = mild compression, ~0.40 = moderate, ~2.50 = severe compression
    lambda_grid: List[float] = field(default_factory=lambda: [0.05, 0.15, 0.40, 1.00, 2.50])
    epochs: int = 15                       # 15 epochs per run on frozen backbones
    batch_size: int = 128                  # Mini-batch size

    # 2. Hardware Regimes to Evaluate
    # Options: ["case_1_uniform", "case_2_capacity_skew", "case_4_dual_skew", "case_3_compute_skew"]
    cases_to_run: List[str] = field(default_factory=lambda: [
        "case_1_uniform",
        "case_2_capacity_skew",
        "case_4_dual_skew",
        "case_3_compute_skew",
    ])

    # 3. Learning Rates (Constant Adam for stable dual exploration)
    lr_s: float = 0.035                    # Gate logits s learning rate
    lr_c: float = 0.025                    # Continuous imputations c learning rate

    # 4. Hard Concrete Mask Relaxation Parameters (Louizos et al., 2018)
    gamma: float = -0.1
    zeta: float = 1.1
    beta: float = 2.0 / 3.0                # Concrete temperature
    alpha: float = 10.0                    # Smooth-max (LSE) sharpness constant
    s_init: float = 2.0                    # Initial gate logit (open transmission)

    # 5. Baseline Reference Channel
    nominal_capacity_mbps: float = 30.0    # 30,000 bits/ms reference channel

    # 6. File & Storage Paths
    root_output_dir: str = "/content/results/pareto_sweep_normalized_benchmark"
    root_drive_dir: str = "/content/drive/MyDrive/SemesterProject_Results/pareto_sweep_normalized_benchmark"
    local_cache_dir: str = "/content/hf_backbones_cache"

SWEEP_CFG = ParetoSweepConfig()

os.makedirs(SWEEP_CFG.root_output_dir, exist_ok=True)
drive_mounted = False
try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    if os.path.exists("/content/drive/MyDrive"):
        os.makedirs(SWEEP_CFG.root_drive_dir, exist_ok=True)
        drive_mounted = True
except Exception:
    pass

TOPO_KEY = "diamond"
weight_path = os.path.join(SWEEP_CFG.local_cache_dir, f"backbone_{TOPO_KEY}.pt")
assert os.path.exists(weight_path), f"Backbone weights missing at {weight_path}. Run Cell 6 first!"

# ------------------------------------------------------------------------------
# 2. Fast GPU Vectorized Caching & Forward Monkey-Patch
# ------------------------------------------------------------------------------
print(">>> Caching CIFAR-10 tensors directly in GPU VRAM...")
cifar_mean = torch.tensor([0.4914, 0.4822, 0.4465], device=device).view(1, 3, 1, 1)
cifar_std = torch.tensor([0.2470, 0.2435, 0.2616], device=device).view(1, 3, 1, 1)

def vectorize_dataset(dataset):
    x = torch.from_numpy(dataset.data).permute(0, 3, 1, 2).to(device, non_blocking=True).float() / 255.0
    x = (x - cifar_mean) / cifar_std
    y = torch.tensor(dataset.targets, dtype=torch.long, device=device)
    return x, y

X_train_gpu, Y_train_gpu = vectorize_dataset(trainset)
X_test_gpu, Y_test_gpu = vectorize_dataset(testset)
N_SAMPLES = X_train_gpu.size(0)

# Memory-optimized forward pass for EdgeBottleneck
def _fast_edge_bottleneck_forward(self, z: torch.Tensor, beta: float, deterministic: bool = False):
    s_broad = self.s.unsqueeze(0)
    c_broad = self.c.unsqueeze(0)
    if not deterministic:
        eps = torch.rand_like(s_broad).clamp(1e-6, 1.0 - 1e-6)
        logits_eps = torch.log(eps) - torch.log(1.0 - eps)
        m_tilde = torch.sigmoid((logits_eps + s_broad) / beta)
        m_bar = m_tilde * (self.gate_cfg.zeta - self.gate_cfg.gamma) + self.gate_cfg.gamma
        m = torch.clamp(m_bar, 0.0, 1.0)
        return z * m + (1.0 - m) * c_broad, None
    else:
        threshold = beta * math.log(-self.gate_cfg.gamma / self.gate_cfg.zeta)
        hard_mask = (s_broad >= threshold).float()
        return z * hard_mask + (1.0 - hard_mask) * c_broad, None

EdgeBottleneck.forward = _fast_edge_bottleneck_forward

# Batched accuracy evaluator
def evaluate_test_accuracy(eval_model, x_test, y_test, beta=0.05, chunk_size=1000):
    eval_model.eval()
    correct = 0
    total = x_test.size(0)
    with torch.no_grad():
        for i in range(0, total, chunk_size):
            bx = x_test[i:i + chunk_size]
            by = y_test[i:i + chunk_size]
            with torch.amp.autocast('cuda'):
                preds = eval_model(bx, beta=beta, deterministic=True)
            correct += (preds.argmax(dim=1) == by).sum().item()
    return 100.0 * correct / total

# ------------------------------------------------------------------------------
# 3. Hardware Regimes
# ------------------------------------------------------------------------------
DIAMOND_CASES = {
    "case_1_uniform": {
        "title": "Case 1: Fully Uniform",
        "nodes": {
            0: NodeConfig("Ingress", exec_time_ms=3.0),
            1: NodeConfig("Worker_A", exec_time_ms=3.0),
            2: NodeConfig("Worker_B", exec_time_ms=3.0),
            3: NodeConfig("Terminal", exec_time_ms=2.0),
        },
        "edges": {
            (0, 1): ChannelConfig.from_mbps(30.0, 1.0),
            (0, 2): ChannelConfig.from_mbps(30.0, 1.0),
            (1, 3): ChannelConfig.from_mbps(30.0, 1.0),
            (2, 3): ChannelConfig.from_mbps(30.0, 1.0),
        },
    },
    "case_2_capacity_skew": {
        "title": "Case 2: Bandwidth Skew",
        "nodes": {
            0: NodeConfig("Ingress", exec_time_ms=3.0),
            1: NodeConfig("Worker_A", exec_time_ms=3.0),
            2: NodeConfig("Worker_B", exec_time_ms=3.0),
            3: NodeConfig("Terminal", exec_time_ms=2.0),
        },
        "edges": {
            (0, 1): ChannelConfig.from_mbps(30.0, 1.0),
            (0, 2): ChannelConfig.from_mbps(30.0, 1.0),
            (1, 3): ChannelConfig.from_mbps(60.0, 1.0),
            (2, 3): ChannelConfig.from_mbps(3.0, 1.0),  # Bottleneck link
        },
    },
    "case_4_dual_skew": {
        "title": "Case 4: Dual Skew (Compute + BW)",
        "nodes": {
            0: NodeConfig("Ingress", exec_time_ms=3.0),
            1: NodeConfig("Worker_A_Fast", exec_time_ms=1.5),
            2: NodeConfig("Worker_B_Slow", exec_time_ms=9.5),
            3: NodeConfig("Terminal", exec_time_ms=2.0),
        },
        "edges": {
            (0, 1): ChannelConfig.from_mbps(30.0, 1.0),
            (0, 2): ChannelConfig.from_mbps(30.0, 1.0),
            (1, 3): ChannelConfig.from_mbps(60.0, 1.0),
            (2, 3): ChannelConfig.from_mbps(3.0, 1.0),  # Compound bottleneck
        },
    },
    "case_3_compute_skew": {
        "title": "Case 3: Compute Delay Skew",
        "nodes": {
            0: NodeConfig("Ingress", exec_time_ms=3.0),
            1: NodeConfig("Worker_A_Fast", exec_time_ms=1.5),
            2: NodeConfig("Worker_B_Slow", exec_time_ms=9.5),
            3: NodeConfig("Terminal", exec_time_ms=2.0),
        },
        "edges": {
            (0, 1): ChannelConfig.from_mbps(30.0, 1.0),
            (0, 2): ChannelConfig.from_mbps(30.0, 1.0),
            (1, 3): ChannelConfig.from_mbps(30.0, 1.0),
            (2, 3): ChannelConfig.from_mbps(3.0, 1.0),
        },
    },
}

PENALTY_METHODS = [
    ("payload_sum", "Omega_1: Payload Sum"),
    ("capacity_scaled", "Omega_2: Capacity-Scaled"),
    ("dag_latency", "Omega_3: Relaxed DAG Latency"),
]

# ------------------------------------------------------------------------------
# 4. Fast Timing Engine & Dynamic-Span [0, 1] Normalized Penalty Function
# ------------------------------------------------------------------------------
class FastDAGTimingEngine(DAGTimingEngine):
    def __init__(self, nodes, edges):
        super().__init__(nodes, edges)
        self.t_exec_tensors = {
            j: torch.tensor(nodes[j].exec_time_ms, device=device, dtype=torch.float32)
            for j in self.nodes
        }
        self.t_prop_tensors = {
            e: torch.tensor(edges[e].prop_delay_ms, device=device, dtype=torch.float32)
            for e in self.edges
        }

    def compute_relaxed_latency(self, bottlenecks, beta: float, alpha: float) -> torch.Tensor:
        t_tilde = {}
        for j in self.topo_order:
            t_exec = self.t_exec_tensors[j]
            if len(self.parents[j]) == 0:
                t_tilde[j] = t_exec
            else:
                branch_arrivals = []
                for i in self.parents[j]:
                    t_trans = bottlenecks[(i, j)].expected_transmission_time_ms(beta)
                    t_prop = self.t_prop_tensors[(i, j)]
                    branch_arrivals.append(t_tilde[i] + t_prop + t_trans)
                stacked = torch.stack(branch_arrivals)
                z_max = torch.max(stacked).detach()
                lse = z_max + (1.0 / alpha) * torch.log(torch.sum(torch.exp(alpha * (stacked - z_max))))
                t_tilde[j] = t_exec + lse
        return t_tilde[self.terminal_node]

C_NOMINAL_BITS_PER_MS = SWEEP_CFG.nominal_capacity_mbps * 1000.0

def compute_normalized_penalty_term(method_key: str, bneck_map: dict, engine: FastDAGTimingEngine,
                                    beta: float, alpha: float, t_floor_const: float, spans: dict):
    """Computes the penalty normalized into [0, 1] based on each method's uncompressed span."""
    if method_key == "dag_latency":
        raw_val = engine.compute_relaxed_latency(bneck_map, beta=beta, alpha=alpha) - t_floor_const
        return raw_val / spans["dag_latency"]
    elif method_key == "capacity_scaled":
        times = [b.expected_transmission_time_ms(beta) for b in bneck_map.values()]
        raw_val = torch.sum(torch.stack(times))
        return raw_val / spans["capacity_scaled"]
    elif method_key == "payload_sum":
        bits = [b.get_transmission_prob(beta).sum() * b.channel_cfg.bit_width for b in bneck_map.values()]
        raw_val = torch.sum(torch.stack(bits)) / C_NOMINAL_BITS_PER_MS
        return raw_val / spans["payload_sum"]

def reset_optimization_state(model_instance, base_checkpoint_path, cfg_obj):
    """Clean reset of s and c parameters and optimizer."""
    loaded_w = torch.load(base_checkpoint_path, map_location=device, weights_only=False)
    model_instance.load_state_dict({k: v for k, v in loaded_w.items() if not (k.endswith(".s") or k.endswith(".c"))}, strict=False)

    for name, p in model_instance.named_parameters():
        if name.endswith(".s") or name.endswith(".c"):
            p.requires_grad = True
            if name.endswith(".s"):
                p.data.fill_(cfg_obj.s_init)
            if name.endswith(".c"):
                p.data.zero_()
        else:
            p.requires_grad = False

    b_map = model_instance.get_bottleneck_map()
    fresh_opt = optim.Adam([
        {"params": [b.s for b in b_map.values()], "lr": cfg_obj.lr_s},
        {"params": [b.c for b in b_map.values()], "lr": cfg_obj.lr_c},
    ], betas=(0.9, 0.999), eps=1e-8)

    return fresh_opt, b_map

# ------------------------------------------------------------------------------
# 5. Measure Baseline Uncompressed Accuracy
# ------------------------------------------------------------------------------
ref_case = DIAMOND_CASES["case_1_uniform"]
gate_cfg_ref = GateConfig(s_init=SWEEP_CFG.s_init, alpha=SWEEP_CFG.alpha, gamma=SWEEP_CFG.gamma, zeta=SWEEP_CFG.zeta)
ref_model = DynamicSplitDAG(TOPO_KEY, gate_cfg_ref, ref_case["edges"]).to(device)
ref_weights = torch.load(weight_path, map_location=device, weights_only=False)
ref_model.load_state_dict({k: v for k, v in ref_weights.items() if not (k.endswith(".s") or k.endswith(".c"))}, strict=False)
for b in ref_model.get_bottleneck_map().values():
    b.s.data.fill_(10.0)

uncompressed_acc = evaluate_test_accuracy(ref_model, X_test_gpu, Y_test_gpu, beta=0.05, chunk_size=1000)
del ref_model
torch.cuda.empty_cache()

print("\n" + "=" * 80)
print(f">>> NORMALIZED PARETO SWEEP SUITE INITIALIZED")
print(f"    Uncompressed Baseline Test Accuracy: {uncompressed_acc:.2f}%")
print(f"    Dimensionless Lambda Grid ({len(SWEEP_CFG.lambda_grid)}): {SWEEP_CFG.lambda_grid}")
print(f"    Epochs per Point: {SWEEP_CFG.epochs} | Gate LR (s): {SWEEP_CFG.lr_s} | Imputation LR (c): {SWEEP_CFG.lr_c}")
print(f"    Cases Selected: {SWEEP_CFG.cases_to_run}")
print("=" * 80)

# ==============================================================================
# 6. Master Sweep Execution Loop across Hardware Regimes
# ==============================================================================
all_sweep_records = []
case_summaries = {}

for case_key in SWEEP_CFG.cases_to_run:
    case_data = DIAMOND_CASES[case_key]
    case_title = case_data["title"]
    case_dir = os.path.join(SWEEP_CFG.root_output_dir, case_key)
    case_drive_dir = os.path.join(SWEEP_CFG.root_drive_dir, case_key) if drive_mounted else None
    os.makedirs(case_dir, exist_ok=True)
    if case_drive_dir:
        os.makedirs(case_drive_dir, exist_ok=True)

    timing_engine = FastDAGTimingEngine(case_data["nodes"], case_data["edges"])
    t_floor = timing_engine.compute_irreducible_latency()

    # Pre-calculate uncompressed ceiling to compute exact dynamic spans
    dummy_gate_cfg = GateConfig(s_init=10.0, alpha=SWEEP_CFG.alpha, gamma=SWEEP_CFG.gamma, zeta=SWEEP_CFG.zeta)
    dummy_model = DynamicSplitDAG(TOPO_KEY, dummy_gate_cfg, case_data["edges"]).to(device)
    for b in dummy_model.get_bottleneck_map().values():
        b.s.data.fill_(10.0)
    bneck_map_open = dummy_model.get_bottleneck_map()
    t_ceil = timing_engine.compute_physical_latency(bneck_map_open, beta=0.05, deterministic=True)
    delta_T = max(t_ceil - t_floor, 1e-4)

    # Dynamic spans for each penalty method
    span_dag = delta_T
    span_capacity = max(sum(b.expected_transmission_time_ms(0.05).item() for b in bneck_map_open.values()), 1e-4)
    total_bits_open = sum(b.get_transmission_prob(0.05).sum().item() * b.channel_cfg.bit_width for b in bneck_map_open.values())
    span_payload = max(total_bits_open / C_NOMINAL_BITS_PER_MS, 1e-4)

    spans_case = {
        "dag_latency": span_dag,
        "capacity_scaled": span_capacity,
        "payload_sum": span_payload,
    }
    del dummy_model
    torch.cuda.empty_cache()

    print(f"\n{'=' * 80}")
    print(f">>> SWEEPING {case_title.upper()}")
    print(f"    Floor: {t_floor:5.2f} ms | Ceiling: {t_ceil:5.2f} ms | Dynamic Span (Delta_T): {delta_T:5.2f} ms")
    print(f"    Normalization Spans -> DAG: {span_dag:5.2f} ms | Cap-Scaled: {span_capacity:5.2f} ms | Payload: {span_payload:5.2f} ms")
    print(f"{'=' * 80}")

    case_records = []

    for method_key, method_label in PENALTY_METHODS:
        for lam_val in SWEEP_CFG.lambda_grid:
            run_name = f"run_{method_key}_lam_{lam_val:.4f}"
            run_dir = os.path.join(case_dir, "runs", run_name)
            run_drive = os.path.join(case_drive_dir, "runs", run_name) if case_drive_dir else None
            os.makedirs(run_dir, exist_ok=True)
            if run_drive:
                os.makedirs(run_drive, exist_ok=True)

            gate_cfg_run = GateConfig(
                s_init=SWEEP_CFG.s_init, alpha=SWEEP_CFG.alpha, gamma=SWEEP_CFG.gamma, zeta=SWEEP_CFG.zeta
            )
            model = DynamicSplitDAG(TOPO_KEY, gate_cfg_run, case_data["edges"]).to(device)
            scaler = torch.amp.GradScaler('cuda')
            crit = nn.CrossEntropyLoss()

            # Clean parameter & optimizer reset
            opt, bneck_map = reset_optimization_state(model, weight_path, SWEEP_CFG)

            # Kinematics buffers
            prev_s = {k: b.s.detach().clone() for k, b in bneck_map.items()}
            prev_c = {k: b.c.detach().clone() for k, b in bneck_map.items()}
            sample_bneck = next(iter(bneck_map.values()))
            gate_thresh = SWEEP_CFG.beta * math.log(-sample_bneck.gate_cfg.gamma / sample_bneck.gate_cfg.zeta)
            prev_masks = {k: (b.s >= gate_thresh).bool().detach().clone() for k, b in bneck_map.items()}
            total_components = sum(m.numel() for m in prev_masks.values())

            traj_records = []

            for ep in range(SWEEP_CFG.epochs):
                model.train()
                perm = torch.randperm(N_SAMPLES, device=device)

                for i in range(0, N_SAMPLES, SWEEP_CFG.batch_size):
                    idx = perm[i:i + SWEEP_CFG.batch_size]
                    x, y = X_train_gpu[idx], Y_train_gpu[idx]
                    opt.zero_grad(set_to_none=True)

                    with torch.amp.autocast('cuda'):
                        y_hat = model(x, beta=SWEEP_CFG.beta, deterministic=False)
                        task_loss = crit(y_hat, y)

                        # Differentiable [0, 1] Normalized Penalty
                        omega_norm = compute_normalized_penalty_term(
                            method_key, bneck_map, timing_engine,
                            SWEEP_CFG.beta, SWEEP_CFG.alpha, t_floor, spans_case
                        )
                        total_loss = task_loss + lam_val * omega_norm

                    scaler.scale(total_loss).backward()
                    scaler.step(opt)
                    scaler.update()

                # --- End-of-Epoch Diagnostics & Kinematics ---
                with torch.no_grad():
                    # 1. Gate s kinematics
                    ds_list = [(b.s - prev_s[k]).abs().flatten() for k, b in bneck_map.items()]
                    all_ds = torch.cat(ds_list)
                    mean_ds = all_ds.mean().item()
                    tail_ds = torch.quantile(all_ds, 0.99).item()

                    # 2. Imputation c kinematics (Velocity & Magnitude)
                    dc_list = [(b.c - prev_c[k]).abs().flatten() for k, b in bneck_map.items()]
                    all_dc = torch.cat(dc_list)
                    mean_dc = all_dc.mean().item()
                    tail_dc = torch.quantile(all_dc, 0.99).item()

                    c_mags = [b.c.abs().flatten() for k, b in bneck_map.items()]
                    all_c_mags = torch.cat(c_mags)
                    mean_abs_c = all_c_mags.mean().item()

                    for k, b in bneck_map.items():
                        prev_s[k].copy_(b.s)
                        prev_c[k].copy_(b.c)

                    # 3. Gate churn & active features
                    curr_masks = {k: (b.s >= gate_thresh).bool() for k, b in bneck_map.items()}
                    total_flips = sum((curr_masks[k] ^ prev_masks[k]).sum().item() for k in bneck_map)
                    total_active = sum(curr_masks[k].sum().item() for k in bneck_map)
                    gate_churn_pct = 100.0 * (total_flips / total_components)
                    active_feature_pct = 100.0 * (total_active / total_components)

                    for k in bneck_map:
                        prev_masks[k].copy_(curr_masks[k])

                    # 4. Latency evaluation & Relaxation Gap
                    acc_pct = evaluate_test_accuracy(model, X_test_gpu, Y_test_gpu, beta=SWEEP_CFG.beta, chunk_size=1000)
                    lat_phys = timing_engine.compute_physical_latency(bneck_map, beta=SWEEP_CFG.beta, deterministic=True)
                    lat_rel = float(timing_engine.compute_relaxed_latency(bneck_map, beta=SWEEP_CFG.beta, alpha=SWEEP_CFG.alpha).item())
                    relax_gap = abs(lat_phys - lat_rel)

                traj_records.append({
                    "epoch": ep + 1,
                    "lam_val": lam_val,
                    "accuracy_pct": acc_pct,
                    "latency_physical_ms": lat_phys,
                    "latency_relaxed_ms": lat_rel,
                    "relaxation_gap_ms": relax_gap,
                    "active_feature_pct": active_feature_pct,
                    "gate_churn_pct": gate_churn_pct,
                    "mean_delta_s": mean_ds,
                    "p99_delta_s": tail_ds,
                    "mean_delta_c": mean_dc,
                    "p99_delta_c": tail_dc,
                    "mean_abs_c": mean_abs_c,
                })

            final_rec = traj_records[-1]
            print(f"  [{method_key:<15} | λ = {lam_val:5.2f}] "
                  f"Physical: {final_rec['latency_physical_ms']:5.2f}ms | Relaxed: {final_rec['latency_relaxed_ms']:5.2f}ms | "
                  f"Gap: {final_rec['relaxation_gap_ms']:4.2f}ms | Acc: {final_rec['accuracy_pct']:5.2f}% | "
                  f"Active%: {final_rec['active_feature_pct']:5.1f}% | E[|Δc|]: {final_rec['mean_delta_c']:6.4f}")

            # Save Trajectory CSV
            df_traj = pd.DataFrame(traj_records)
            df_traj.to_csv(os.path.join(run_dir, "trajectory.csv"), index=False)
            if run_drive:
                shutil.copy(os.path.join(run_dir, "trajectory.csv"), os.path.join(run_drive, "trajectory.csv"))

            # ------------------------------------------------------------------
            # 4-Panel Comprehensive Kinematics Diagnostic Plot
            # ------------------------------------------------------------------
            ep_axis = df_traj["epoch"]
            fig_run, axes = plt.subplots(1, 4, figsize=(23, 4.8), dpi=140)

            # Panel (a): Latency Trajectory & Relaxation Gap
            ax0_left = axes[0]
            ax0_right = ax0_left.twinx()
            l1 = ax0_left.plot(ep_axis, df_traj["latency_physical_ms"], "o-", color="#D62728", lw=2.2, ms=4, label="Physical (Det)", zorder=3)
            l2 = ax0_left.plot(ep_axis, df_traj["latency_relaxed_ms"], "--", color="#FF7F0E", lw=1.8, label="Relaxed (Surrogate)", zorder=3)
            l3 = ax0_left.axhline(t_floor, color="black", linestyle="-", lw=1.2, label=f"Floor ({t_floor:.1f}ms)", zorder=2)
            l4 = ax0_left.axhline(t_ceil, color="black", linestyle=":", lw=1.2, label=f"Ceiling ({t_ceil:.1f}ms)", zorder=2)
            l5 = ax0_right.plot(ep_axis, df_traj["relaxation_gap_ms"], ":", color="gray", lw=1.5, label="Gap |t_phys - t_rel|", zorder=2)
            ax0_left.set_xlabel("Epoch", fontweight="bold")
            ax0_left.set_ylabel("Latency (ms)", fontweight="bold")
            ax0_right.set_ylabel("Relaxation Gap (ms)", color="gray", fontweight="bold")
            ax0_left.set_title(f"(a) Latency & Relaxation Gap\n{method_label} (λ = {lam_val})", fontweight="bold")
            ax0_left.grid(True, linestyle=":", alpha=0.6)
            lines_a = l1 + l2 + [l3, l4] + l5
            ax0_left.legend(lines_a, [line.get_label() for line in lines_a], loc="center right", fontsize=7.2)

            # Panel (b): Test Accuracy Evolution
            axes[1].plot(ep_axis, df_traj["accuracy_pct"], "s-", color="#1F77B4", lw=2.2, ms=4, label="Trained Split Model", zorder=3)
            axes[1].axhline(uncompressed_acc, color="darkgreen", linestyle="--", lw=1.5, label=f"Uncompressed Base ({uncompressed_acc:.1f}%)", zorder=2)
            axes[1].set_xlabel("Epoch", fontweight="bold")
            axes[1].set_ylabel("CIFAR-10 Accuracy (%)", fontweight="bold")
            axes[1].set_title("(b) Test Accuracy Evolution", fontweight="bold")
            axes[1].grid(True, linestyle=":", alpha=0.6)
            axes[1].legend(loc="lower right", fontsize=8.0)

            # Panel (c): Gate (s) Kinematics & Active Features
            ax2_left = axes[2]
            ax2_right = ax2_left.twinx()
            ls1 = ax2_left.plot(ep_axis, df_traj["mean_delta_s"], "g-", lw=1.8, label="E[|Δs|] (Velocity)", zorder=3)
            ls2 = ax2_left.plot(ep_axis, df_traj["p99_delta_s"], "g--", lw=1.4, alpha=0.7, label="p99(|Δs|) (Tail)", zorder=3)
            ls3 = ax2_right.plot(ep_axis, df_traj["active_feature_pct"], "m:", lw=2.0, label="Active Features (%)", zorder=3)
            ax2_left.set_yscale("log")
            ax2_left.set_xlabel("Epoch", fontweight="bold")
            ax2_left.set_ylabel("Gate Velocity |Δs|", color="green", fontweight="bold")
            ax2_right.set_ylabel("Active Features (%)", color="purple", fontweight="bold")
            ax2_left.set_title("(c) Gate (s) Velocity & Pruning", fontweight="bold")
            ax2_left.grid(True, linestyle=":", alpha=0.6)
            lines_c = ls1 + ls2 + ls3
            ax2_left.legend(lines_c, [line.get_label() for line in lines_c], loc="upper right", fontsize=7.5)

            # Panel (d): Imputation (c) Kinematics & Magnitude
            ax3_left = axes[3]
            ax3_right = ax3_left.twinx()
            lc1 = ax3_left.plot(ep_axis, df_traj["mean_delta_c"], "b-", lw=1.8, label="E[|Δc|] (Velocity)", zorder=3)
            lc2 = ax3_left.plot(ep_axis, df_traj["p99_delta_c"], "b--", lw=1.4, alpha=0.7, label="p99(|Δc|) (Tail)", zorder=3)
            lc3 = ax3_right.plot(ep_axis, df_traj["mean_abs_c"], "k-.", lw=1.8, label="E[|c|] (Mean Magnitude)", zorder=3)
            ax3_left.set_yscale("log")
            ax3_left.set_xlabel("Epoch", fontweight="bold")
            ax3_left.set_ylabel("Imputation Velocity |Δc|", color="blue", fontweight="bold")
            ax3_right.set_ylabel("Magnitude E[|c|]", color="black", fontweight="bold")
            ax3_left.set_title("(d) Imputation (c) Drift & Velocity", fontweight="bold")
            ax3_left.grid(True, linestyle=":", alpha=0.6)
            lines_d = lc1 + lc2 + lc3
            ax3_left.legend(lines_d, [line.get_label() for line in lines_d], loc="upper right", fontsize=7.5)

            plt.tight_layout()
            plt.savefig(os.path.join(run_dir, "kinematics.pdf"), format="pdf", bbox_inches="tight")
            plt.savefig(os.path.join(run_dir, "kinematics.png"), format="png", dpi=180, bbox_inches="tight")
            if run_drive:
                shutil.copy(os.path.join(run_dir, "kinematics.pdf"), os.path.join(run_drive, "kinematics.pdf"))
                shutil.copy(os.path.join(run_dir, "kinematics.png"), os.path.join(run_drive, "kinematics.png"))
            plt.close(fig_run)

            summary_pt = {
                "case_key": case_key,
                "case_title": case_title,
                "method_key": method_key,
                "method_label": method_label,
                "lam_val": lam_val,
                "achieved_physical_ms": final_rec["latency_physical_ms"],
                "achieved_relaxed_ms": final_rec["latency_relaxed_ms"],
                "relaxation_gap_ms": final_rec["relaxation_gap_ms"],
                "final_accuracy_pct": final_rec["accuracy_pct"],
                "active_feature_pct": final_rec["active_feature_pct"],
                "final_churn_pct": final_rec["gate_churn_pct"],
                "final_mean_ds": final_rec["mean_delta_s"],
                "final_mean_dc": final_rec["mean_delta_c"],
                "final_mean_abs_c": final_rec["mean_abs_c"],
            }
            case_records.append(summary_pt)
            all_sweep_records.append(summary_pt)

    # --------------------------------------------------------------------------
    # Case-Level Pareto Frontier Plot
    # --------------------------------------------------------------------------
    df_case = pd.DataFrame(case_records)
    df_case.to_csv(os.path.join(case_dir, "case_pareto_summary.csv"), index=False)
    if case_drive_dir:
        shutil.copy(os.path.join(case_dir, "case_pareto_summary.csv"), os.path.join(case_drive_dir, "case_pareto_summary.csv"))
    case_summaries[case_key] = df_case

    fig_case, ax = plt.subplots(figsize=(8.5, 5.5), dpi=150)
    colors = {"payload_sum": "#D62728", "capacity_scaled": "#FF7F0E", "dag_latency": "#1F77B4"}
    markers = {"payload_sum": "^", "capacity_scaled": "s", "dag_latency": "o"}

    ax.axvspan(t_floor, t_ceil, color="#EAEAF2", alpha=0.5, label="Compressible Dynamic Span")
    ax.axvline(t_floor, color="black", linestyle="-", lw=1.6, label=f"Floor ({t_floor:.1f}ms)")
    ax.axvline(t_ceil, color="black", linestyle="--", lw=1.6, label=f"Ceiling ({t_ceil:.1f}ms)")
    ax.axhline(uncompressed_acc, color="darkgreen", linestyle=":", lw=1.4, label=f"Uncompressed Base ({uncompressed_acc:.1f}%)")

    for m_key, m_label in PENALTY_METHODS:
        sub = df_case[df_case["method_key"] == m_key].sort_values("achieved_physical_ms")
        ax.plot(
            sub["achieved_physical_ms"], sub["final_accuracy_pct"],
            marker=markers[m_key], color=colors[m_key], lw=2.2, ms=7, label=m_label
        )
        for _, row in sub.iterrows():
            ax.annotate(
                f"λ={row['lam_val']}",
                (row["achieved_physical_ms"], row["final_accuracy_pct"]),
                textcoords="offset points", xytext=(0, 6), ha="center", fontsize=7.5, color=colors[m_key]
            )

    ax.set_xlabel("Physical Latency (ms)", fontsize=11, fontweight="bold")
    ax.set_ylabel("CIFAR-10 Test Accuracy (%)", fontsize=11, fontweight="bold")
    ax.set_title(f"Pareto Frontier: {case_title}\nAccuracy vs. Physical Latency ([0, 1] Normalized Penalties)", fontsize=12, fontweight="bold")
    ax.grid(True, linestyle=":", alpha=0.6)
    ax.legend(loc="lower right", fontsize=8.5)
    plt.tight_layout()

    plt.savefig(os.path.join(case_dir, "pareto_frontier.pdf"), format="pdf", bbox_inches="tight")
    plt.savefig(os.path.join(case_dir, "pareto_frontier.png"), format="png", dpi=220, bbox_inches="tight")
    if case_drive_dir:
        shutil.copy(os.path.join(case_dir, "pareto_frontier.pdf"), os.path.join(case_drive_dir, "pareto_frontier.pdf"))
        shutil.copy(os.path.join(case_dir, "pareto_frontier.png"), os.path.join(case_drive_dir, "pareto_frontier.png"))
    plt.close(fig_case)

# ==============================================================================
# 7. Joint 4-Case Publication Pareto Figure
# ==============================================================================
print("\n" + "=" * 80)
print(">>> COMPILING FINAL 4-CASE JOINT PUBLICATION FIGURE...")
print("=" * 80)

df_all = pd.DataFrame(all_sweep_records)
df_all.to_csv(os.path.join(SWEEP_CFG.root_output_dir, "all_cases_pareto_data.csv"), index=False)
if drive_mounted:
    shutil.copy(os.path.join(SWEEP_CFG.root_output_dir, "all_cases_pareto_data.csv"),
                os.path.join(SWEEP_CFG.root_drive_dir, "all_cases_pareto_data.csv"))

fig_joint, axes = plt.subplots(2, 2, figsize=(16, 11), dpi=160)
axes_flat = axes.flatten()

colors = {"payload_sum": "#D62728", "capacity_scaled": "#FF7F0E", "dag_latency": "#1F77B4"}
markers = {"payload_sum": "^", "capacity_scaled": "s", "dag_latency": "o"}

for idx, case_key in enumerate(SWEEP_CFG.cases_to_run):
    ax = axes_flat[idx]
    case_data = DIAMOND_CASES[case_key]
    c_title = case_data["title"]
    df_c = case_summaries[case_key]

    c_engine = FastDAGTimingEngine(case_data["nodes"], case_data["edges"])
    f_lat = c_engine.compute_irreducible_latency()

    dummy_m = DynamicSplitDAG(TOPO_KEY, dummy_gate_cfg, case_data["edges"]).to(device)
    for b in dummy_m.get_bottleneck_map().values():
        b.s.data.fill_(10.0)
    c_lat = c_engine.compute_physical_latency(dummy_m.get_bottleneck_map(), beta=0.05, deterministic=True)
    del dummy_m

    ax.axvspan(f_lat, c_lat, color="#EAEAF2", alpha=0.5)
    ax.axvline(f_lat, color="black", linestyle="-", lw=1.4, alpha=0.8, label=f"Floor ({f_lat:.1f}ms)")
    ax.axvline(c_lat, color="black", linestyle="--", lw=1.4, alpha=0.8, label=f"Ceiling ({c_lat:.1f}ms)")
    ax.axhline(uncompressed_acc, color="darkgreen", linestyle=":", lw=1.2, alpha=0.7)

    for m_key, m_label in PENALTY_METHODS:
        sub = df_c[df_c["method_key"] == m_key].sort_values("achieved_physical_ms")
        ax.plot(
            sub["achieved_physical_ms"], sub["final_accuracy_pct"],
            marker=markers[m_key], color=colors[m_key], lw=2.2, ms=6, label=m_label
        )

    ax.set_xlabel("Physical Latency (ms)", fontsize=10.5, fontweight="bold")
    ax.set_ylabel("CIFAR-10 Accuracy (%)", fontsize=10.5, fontweight="bold")
    ax.set_title(f"({chr(97+idx)}) {c_title}", fontsize=11.5, fontweight="bold")
    ax.grid(True, linestyle=":", alpha=0.6)
    if idx == 0:
        ax.legend(loc="lower right", fontsize=8.5)

plt.suptitle(
    "Pareto Frontier Benchmark: End-to-End DAG Latency vs. Channel & Payload Surrogates\n"
    f"[Uncompressed Baseline Accuracy: {uncompressed_acc:.2f}% | Dynamic Span [0, 1] Normalized Mode]",
    fontsize=13.5,
    fontweight="bold",
    y=0.995,
)
plt.tight_layout()

joint_pdf = os.path.join(SWEEP_CFG.root_output_dir, "joint_pareto_comparison.pdf")
joint_png = os.path.join(SWEEP_CFG.root_output_dir, "joint_pareto_comparison.png")
plt.savefig(joint_pdf, format="pdf", bbox_inches="tight")
plt.savefig(joint_png, format="png", dpi=250, bbox_inches="tight")

if drive_mounted:
    shutil.copy(joint_pdf, os.path.join(SWEEP_CFG.root_drive_dir, "joint_pareto_comparison.pdf"))
    shutil.copy(joint_png, os.path.join(SWEEP_CFG.root_drive_dir, "joint_pareto_comparison.png"))
    print(f"\n>>> [SUCCESS] All figures, plots, and CSV summaries synced to Drive:\n    {SWEEP_CFG.root_drive_dir}")

plt.show()
plt.close(fig_joint)

print("\n" + "=" * 80)
print(">>> NORMALIZED PARETO SWEEP SUITE COMPLETE!")
print("=" * 80)

Mounted at /content/drive
>>> Caching CIFAR-10 tensors directly in GPU VRAM...

>>> NORMALIZED PARETO SWEEP SUITE INITIALIZED
    Uncompressed Baseline Test Accuracy: 91.88%
    Dimensionless Lambda Grid (5): [0.05, 0.15, 0.4, 1.0, 2.5]
    Epochs per Point: 15 | Gate LR (s): 0.035 | Imputation LR (c): 0.025
    Cases Selected: ['case_1_uniform', 'case_2_capacity_skew', 'case_4_dual_skew', 'case_3_compute_skew']

>>> SWEEPING CASE 1: FULLY UNIFORM
    Floor: 10.00 ms | Ceiling: 23.11 ms | Dynamic Span (Delta_T): 13.11 ms
    Normalization Spans -> DAG: 13.11 ms | Cap-Scaled: 26.21 ms | Payload: 26.21 ms
  [payload_sum     | λ =  0.05] Physical: 22.70ms | Relaxed: 22.63ms | Gap: 0.08ms | Acc: 91.22% | Active%:  96.9% | E[|Δc|]: 0.2555
  [payload_sum     | λ =  0.15] Physical: 22.46ms | Relaxed: 22.41ms | Gap: 0.05ms | Acc: 91.07% | Active%:  95.0% | E[|Δc|]: 0.2507
  [payload_sum     | λ =  0.40] Physical: 21.76ms | Relaxed: 21.72ms | Gap: 0.04ms | Acc: 90.93% | Active%:  89.3% | E[|Δc|